# 🤖 **Modern AI Foundations — Build with LLMs, Agents, RAG & LoRA** 🧠

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaopanboonyuen/WALDO2026/blob/main/code/LLM_AI_NSTDA2026_toStudent.ipynb)

![Runs on Colab T4 GPU](https://img.shields.io/badge/Colab-T4%20GPU%20Free-orange?logo=googlecolab)

![Hugging Face](https://img.shields.io/badge/Hugging%20Face-Transformers-yellow?logo=huggingface)

![PyTorch](https://img.shields.io/badge/PyTorch-GPU-red?logo=pytorch)

![PEFT](https://img.shields.io/badge/PEFT-LoRA-blueviolet)

### 🚀 NSTDA AI Workshop 2026 · by Dr. Teerapong Panboonyuen (P'Kao)

You've heard about **LLMs, multimodal AI, agents, RAG, and fine-tuning**.

Now let's **actually run them**.

This notebook is a compact hands-on tour of modern AI foundations. Instead of spending the whole session discussing how these systems work, you'll load a real open LLM, make it generate text, connect AI to images, build a tiny tool-using agent, ground answers with RAG, and finally peek inside **LoRA fine-tuning**.

Everything is designed to run on a **free Google Colab T4 GPU** with small models and short experiments.

> 🧠 **The goal isn't to build GPT-5 today.**
>
> The goal is to understand the fundamental building blocks behind modern AI systems by **running each idea yourself**.

---

## 🗺️ **Today's Roadmap**

| Step | Concept | What we'll do |
|---|---|---|
| 0 | 🔧 **Setup** | Prepare Colab, install Transformers / PEFT / Sentence Transformers |
| 1 | 💬 **LLM** | Load and chat with **Qwen2.5-0.5B-Instruct** |
| 2 | 🖼️ **Multimodal AI** | Use **CLIP** for zero-shot image understanding |
| 3 | 🛠️ **Agentic AI** | Build a tiny agent that can decide when to call a calculator |
| 4 | 📚 **RAG** | Retrieve relevant knowledge before asking the LLM to answer |
| 5 | 🪶 **Fine-Tuning / LoRA** | See how few parameters LoRA actually needs to train |
| 6 | 🎓 **Your Turn** | Modify the prompts, experiment, and break things! |

The notebook deliberately keeps these experiments small so that the models can load and run quickly on a free T4 GPU.

---

## 0. Setup 🔧

We need `transformers` (to load foundation models), `accelerate` (fast loading), `peft` (for the LoRA peek in Section 5), and `sentence-transformers` (for the RAG retriever in Section 4). Colab already has `torch`, `numpy`, `matplotlib`, `Pillow`, and `requests`, so we only install what's missing.

In [ ]:
# Run this once per Colab session — takes about 20-30 seconds
!pip -q install insert-your-llm-libraries --upgrade

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import torch
import requests
from PIL import Image
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU, then Run all again.")

## 1. The LLM Concept, Running Live 💬

We'll load **Qwen2.5-0.5B-Instruct** — a genuinely small (0.5-billion-parameter) open LLM. It is nowhere near as capable as GPT-4o or Claude, but it is a *real* Transformer, trained the *real* way (pretrain → instruction fine-tune), and it is small enough to download and run in seconds. That trade-off — capability vs. size/speed — is exactly the kind of decision you'll make when picking a foundation model for a real project.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "INSERT YOUR FIRST FOUNDATION MODEL"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=DTYPE).to(DEVICE)
llm.eval()

print(f"Loaded {LLM_NAME}")
print(f"Parameters: {sum(p.numel() for p in llm.parameters()):,}")

In [ ]:
def chat(user_message, system_message="You are a concise, helpful assistant.", max_new_tokens=200):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print(chat("Explain what an AI agent is, in exactly two sentences."))

In [ ]:
# Try your own prompt! A small model like this is fun for short, playful requests too.
print(chat("Insert your own prompt!"))

> 💡 **Try it yourself:** change the prompt in the cell above (or add a new cell) and re-run. Notice the response is *sampled* — running the same prompt twice can give slightly different wording, because `do_sample=True`.

## 2. Multimodal AI, Running Live 🖼️

**CLIP** (Contrastive Language–Image Pre-training) is one of the earliest and simplest multimodal models: it embeds *images* and *text* into the **same** vector space, so you can ask "how well does this image match this sentence?" — with zero task-specific training. This is called **zero-shot classification**: we never trained CLIP to recognize these exact labels, yet it can rank them for a brand-new image.

In [ ]:
from transformers import INSERT_YOUR_FM_METHODS

CLIP_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)

image_url = "https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/raw/main/img/sample_cat_image_from_val2017.jpg"  # classic CLIP demo photo
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis("off")
plt.title("What does the model see?")
plt.show()

In [ ]:
candidate_labels = ["a photo of two cats", "a photo of a dog", "a photo of a laptop", "a plate of food"]

inputs = clip_processor(text=candidate_labels, images=image, return_tensors="pt", padding=True).to(DEVICE)

with torch.no_grad():
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[0].cpu().numpy()

for label, prob in sorted(zip(candidate_labels, probs), key=lambda x: -x[1]):
    bar = "█" * int(prob * 40)
    print(f"{prob*100:5.1f}%  {bar}  {label}")

> 💡 **Try it yourself:** change `candidate_labels` to something food- or product-related, or swap `image_url` for any public image link, and re-run. This is the same underlying idea behind "search my photos for X" and behind the image half of models like GPT-4o and Claude with vision.

## 3. Agentic AI, Running Live 🛠️

An **agent** doesn't just answer — it can decide to use a **tool**, look at the tool's result, and *then* answer. Below we give our small LLM exactly one tool: a calculator. We tell it the format to ask for the tool in, let it decide whether it needs it, and if it does, we execute the calculation ourselves and hand the result back — that's one full turn of the **observe → think → act** loop from the lecture.

> ⚠️ Small, 0.5B-parameter models don't follow instructions as reliably as the large ones (GPT-4o, Claude, Gemini) you'd actually build production agents with. If the model doesn't use the exact format below, the code just falls back to its direct answer and tells you so — that inconsistency *is* the lesson: agent reliability is a real engineering problem, not just a prompting trick.

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a helpful assistant with access to ONE tool.

Tool: calculator(expression) -> evaluates a basic arithmetic expression and returns the numeric result.

If answering the user requires arithmetic, respond with EXACTLY one line, nothing else:
ACTION: calculator("<expression>")
Do not compute the arithmetic yourself when you use the tool.

If you do NOT need the tool, just answer normally in one short sentence.

Example:
User: What is 12 times 8?
Assistant: ACTION: calculator("12 * 8")
"""

def safe_calculator(expression):
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expression):
        raise ValueError("Unsafe expression blocked.")
    return eval(expression, {"__builtins__": {}}, {})


def run_agent(question, max_new_tokens=60):
    print(f"USER QUESTION: {question}\n")

    step1 = chat(question, system_message=AGENT_SYSTEM_PROMPT, max_new_tokens=max_new_tokens)
    print(f"[Step 1 — Think] Model said: {step1!r}\n")

    match = re.search(r'ACTION:\s*calculator\("([^"]+)"\)', step1)
    if not match:
        print("[No tool call detected — treating this as the final answer]")
        print(f"\nFINAL ANSWER: {step1}")
        return

    expression = match.group(1)
    try:
        result = safe_calculator(expression)
        print(f"[Step 2 — Act] Running calculator(\"{expression}\") -> {result}\n")
    except Exception as e:
        print(f"[Step 2 — Act] Tool call failed: {e}")
        return

    followup = (
        f"The original question was: {question}\n"
        f"OBSERVATION: calculator(\"{expression}\") = {result}\n"
        f"Now give the final answer to the user's original question in ONE short sentence, using this result."
    )
    step3 = chat(followup, system_message="You are a helpful assistant.", max_new_tokens=60)
    print(f"[Step 3 — Observe + Final Answer] {step3}")


run_agent("A ticket costs 350 baht. I'm buying 4 tickets, and there is a 15% discount on the total. How much do I pay?")

> 💡 **Try it yourself:** ask something that needs no arithmetic (e.g. `"What's a good name for a coffee shop?"`) and confirm the agent skips the tool. Then try a harder calculation and see whether the model formats the tool call correctly every time.

## 4. RAG, Running Live 📚

Instead of hoping the LLM "remembers" a fact, **RAG** retrieves the most relevant piece of text from a knowledge base *first*, and only then asks the LLM to answer — using that retrieved text as grounding. Our "knowledge base" here is five one-line facts about this very course.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

knowledge_base = [
    "CP020003 Artificial Intelligence is taught at Khon Kaen University in 2026.",
    "The midterm exam covers Weeks 1 through 7: from feature engineering to association rule mining.",
    "LoRA (Low-Rank Adaptation) freezes the pretrained weights and only trains small added matrices.",
    "RAG stands for Retrieval-Augmented Generation and grounds an LLM's answer in retrieved documents.",
    "An AI agent repeats an observe-think-act loop until it reaches its goal, unlike a single-turn chatbot.",
]

doc_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True, normalize_embeddings=True)
print(f"Embedded {len(knowledge_base)} facts into {doc_embeddings.shape[1]}-dimensional vectors.")

In [ ]:
def retrieve(question, top_k=1):
    q_embedding = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    similarities = (doc_embeddings @ q_embedding.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(-similarities)[:top_k]
    return [(knowledge_base[i], float(similarities[i])) for i in top_idx]


def rag_answer(question, top_k=1):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n".join(f"- {text}" for text, score in retrieved)

    print("RETRIEVED CONTEXT:")
    for text, score in retrieved:
        print(f"  (similarity {score:.3f}) {text}")
    print()

    prompt = (
        f"Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    answer = chat(prompt, system_message="You are a precise assistant that only uses the given context.", max_new_tokens=80)
    print(f"GROUNDED ANSWER: {answer}")


rag_answer("Which weeks does the midterm cover?")

> 💡 **Try it yourself:** ask `rag_answer("What does LoRA freeze?")` or a question with *no* matching fact (e.g. `"Who teaches Week 12?"`) and see whether the model correctly says it doesn't know instead of guessing.

## 5. Fine-Tuning / LoRA, Running Live 🪶

We won't run a full training job — that needs a real dataset and more than a minute of GPU time. But we *can* show, in seconds, the exact number that makes LoRA attractive: how many parameters it actually trains compared to the full model. We wrap our LLM with a LoRA adapter (the same `peft` library used in real fine-tuning jobs) and ask it to report the difference.

In [ ]:
!pip install -q -U "torchao>=0.16.0" peft

In [ ]:
from peft import LoraConfig, get_peft_model

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("torchao").setLevel(logging.ERROR)


total_params = sum(p.numel() for p in llm.parameters())

lora_config = LoraConfig(
    r=8,                                    # rank of the small A/B matrices from the lecture diagram
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],    # which layers get an adapter
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lora_llm = get_peft_model(llm, lora_config)
lora_llm.print_trainable_parameters()

trainable_params = sum(p.numel() for p in lora_llm.parameters() if p.requires_grad)
print(f"\nFull model parameters:      {total_params:,}")
print(f"LoRA-trainable parameters:  {trainable_params:,}")
print(f"That's only {trainable_params / total_params * 100:.3f}% of the model being trained.")

> 💡 **What just happened:** every one of those ~500 million original weights is still there and still frozen — `get_peft_model` only *added* a small number of new, trainable matrices next to a couple of layers. That tiny fraction is why LoRA fine-tuning jobs that would otherwise need an expensive multi-GPU server can often run on a single free-tier GPU like this one.

## Recap — What You Just Ran ✅

| # | Concept | What ran | Model used |
|---|---|---|---|
| 1 | LLM | A real chat conversation | Qwen2.5-0.5B-Instruct |
| 2 | Multimodal AI | Zero-shot image classification | CLIP (ViT-B/32) |
| 3 | Agentic AI | Observe → think → act loop with a tool | Qwen2.5-0.5B-Instruct |
| 4 | RAG | Retrieval + grounded generation | MiniLM embeddings + Qwen2.5-0.5B-Instruct |
| 5 | Fine-tuning / LoRA | Parameter-efficient adapter setup | peft + Qwen2.5-0.5B-Instruct |

All five ran on a **free** Colab GPU, in a few minutes total — that's the entire point. Foundation models are no longer something only a big lab can touch; they're something you can load, poke, and break in class.

---

### 🍀 Before we go — Good luck on your AI journey 
if you choose the path of becoming an AI engineer.

P'Kao